Python version 확인 및 langchain  및 library 설치

(tutorial 내용 출처 : https://wikidocs.net/231393)

In [ ]:
!python --version

In [ ]:
!pip install langchain>=1.0.0
!pip install langchain-core>=1.0.0
!pip install langchain-openai
!pip install langchain langchain-anthropic langgraph
!pip install tiktoken
!pip install langchain-community
!pip install unstructured
!pip install -q pypdf
!pip install chromadb
!pip install -U sentence-transformers

In [ ]:
from langchain_openai import ChatOpenAI
import os
import google.colab import userdata
import openai

# API 키 설정하기.
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# ChatOpenAI 모델 초기화하기.
llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature = 0.7
)

langchain 관련 version 확인

In [ ]:
import langchain
import langchain_core
import langchain_openai
import langchain_community
print(f"Langchain 버전 : {langchain.__version__}")
print(f"Langchain-core 버전 : {langchain_core.__version__}")

In [ ]:
!pip show langchain_openai

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print("[OK] API 키 설정됨"if os.getenv("OPENAI_API_KEY") else "[ERROR] API 키 미설정")



OPENAI에서 결제 완료.
API_KEY 발급 완료.
Google colab의 secret key에 등록 완료.
위의 내용 확인 및 test 진행.

In [ ]:
from google.colab import userdata
import os

try:
  os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
  print("[OK] API key load done")
except Exception:
  print("[WARN] colab secret에 OPENAI_API_KEY 등록하기.")

LLMChain 만들기.
LCEL(LangChain Expression Language) 사용.

LLM chain == prompt + LLM
사용자의 입력(prompt) 받아 LLM 통해 적절한 응답이나 결과를 생성하는 구조.

- prompt : 사용자 또는 system에서 제공하는 입력, LLM에게 특정 작업을 수행하도록 요청하는 지시문.
- LLM : GPT, Gemini 등 대규모 언어 모델로 prompt 바탕으로 적절한 응답 생성 / 주어진 작업 수행.

일반적인 작동 방식
- prompt 생성
- LLM 처리
- 응답 반환

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "{topic}에 대해 간단히 설명해주세요."
)

llm = init_chat_model("gpt-4o-mini")

# chain 구성(prompt | LLM)
chain = prompt | llm

response = chain.invoke({"topic" : "한국외대"})
print(response.content)
# 결과적으로 topic에 대한 gpt-4o-mini의 답변을 확인할 수 있음.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

#chain 구성에 출력 parser 추가.
chain = prompt | llm | StrOutputParser()

# 문자열로 바로 결과를 반환.
# 즉, 위의 코드와 실행 결과는 동일하나 response.content가 아닌 문자열로 바로 답변을 받을 수 있음.
# 이때, chain.invoke()는 단일 입력에 대한 함수
result = chain.invoke({"topic" : "한국외대"})
print(result)

In [ ]:
# chain.batch()는 여러 입력을 병렬 처리할 때 사용.
results = chain.batch([
    {"topic" : "한국외대"},
    {"topic" : "한국외대 컴퓨터공학부"},
    {"topic" : "HUFS"}
])

for result in results:
  print(result)
  print("\n------------------------------------\n")

기본 LLM chain

In [ ]:
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.prompts.chat import HumanMessagePromptTemplate
#prompt2 : 수학 전문가로서 질문에 답변하는 형식의 prompt template
#prompt2 사용하면 입력으로 주어진 질문을 {input}에 넣어서 수학 전문가의 관점에서 답변을 생성하는 질문 prompt 만들 수 있음.
prompt2 = ChatPromptTemplate.from_template("You are an expert in math. Answer the question. <Question> : {input}")
# 수학 전문가처럼 행동하라고 model에게 지시함.
prompt2

ChatPromptTemplate(
    input_variables=['input'],
    messages=[HumanMessagePromptTemplate(
        prompt=PromptTemplate(
            input_variables=['input'],
            template='You are an expert in math. Answer the question. <Question> : {input}'
            )
        )
    ]
)

chain = prompt2 | llm | StrOutputParser()
# 새로운 prompt 만들었기 때문에 chain 연결을 갱신해야 함.
chain.invoke({"input": "원주율은?"})


Multi-Chain


---


Multi-Chain Pattern
- 순차적 Chain(Sequential)
   - chain = A | B | C
   - 흐름도 : A -> B -> C
   - 목적 : 단계별 처리가 필요할 때.
- 병렬 Chain(Parallel)
   - chain = RunnableParallel(a=A,b=B)
   - 흐름도 : A와 B 동시 실행 및 결과 병합
   - 목적 : 독립적 작업을 동시에 실행할 때.
  - 조건부 분기(Branching)
    - chain = RunnableBranch(...)
    - 흐름도 : 조건에 따라 A 또는 B 실행
    - 목적 : 입력에 다라 다른 처리를 진행할 때.
  - 복합
    - 위 Pattern을 조합해서 사용하는 Pattern.
    - 목적 : RAG pipeline 등에 사용.

In [ ]:
# 순차적 Chain 연결 - 기본 순차 Chain

# translate_prompt 먼저 실행 => 영어로 번역.
# translate_prompt의 실행 결과인 영어 단어를 한국어로 설명하는 explain_prompt
# 순차 Chain(translate_prompt => explain_prompt)
translate_prompt = ChatPromptTemplate.from_template(
    "다음 한국어를 영어로 번역하시오: {korean_word}"
)

explain_prompt = ChatPromptTemplate.from_template(
    "다음 영어 단어를 한국어로 자세히 설명하시오: {english_word}"
)

chain1 = translate_prompt | llm | StrOutputParser()

# chain1의 결과(=translate_prompt의 결과)를 입력으로 사용.
chain2 = (
    {"english_word" : chain1}
    | explain_prompt
    | llm
    | StrOutputParser()
)

result = chain2.invoke({"korean_word" : "인공지능"})
print(result)

In [ ]:
# 순차적 Chain 연결 = 다단계 순차 Chain
from langchain_core.runnables import RunnablePassthrough

analyze_prompt = ChatPromptTemplate.from_template(
    "다음 주제의 핵심 키워트 3개를 추출하세요: {topic}"
)

outline_prompt = ChatPromptTemplate.from_template(
    """다음 키워드를 바탕으로 글의 개요를 작성하세요:
    키워드 : {keywords}
    원본 주제 : {topic}"""
)

content_prompt = ChatPromptTemplate.from_template(
    """다음 개요를 바탕으로 300자 내외의 글을 작성하세요:
    개요: {outline}"""
)

chain = (
    {"topic" : RunnablePassthrough()}
    | RunnablePassthrough.assign(
        keywords=analyze_prompt | llm | StrOutputParser()
    )
    | RunnablePassthrough.assign(
        outline=outline_prompt | llm | StrOutputParser()
    )
    | content_prompt
    |llm
    | StrOutputParser()
)

result = chain.invoke({"topic" : "기후 변화와 지속 가능한 발전"})
print(result)

In [ ]:
#병렬 Chian - RunnableParallel 기본 사용
from langchain_core.runnables import RunnableParallel

positive_prompt = ChatPromptTemplate.from_template(
    "{topic}의 긍정적인 측면 3가지를 설명하세요."
)

negative_prompt = ChatPromptTemplate.from_template(
    "{topic}의 부정적인 측면 3가지를 설명하세요."
)

neutral_prompt = ChatPromptTemplate.from_template(
    "{topic}에 대한 객관적인 현황을 설명하세요."
)

# 병렬 Chain 구성
parallel_chain = RunnableParallel(
    positive=positive_prompt | llm | StrOutputParser(),
    negative=negative_prompt | llm | StrOutputParser(),
    neutral=neutral_prompt | llm | StrOutputParser(),
)

# 세 가지 chain 동시에 실행.
results = parallel_chain.invoke({"topic" : "원격 근무"})

print(results["positive"] + "\n\n")
print(results["negative"] + "\n\n")
print(results["neutral"] + "\n\n")

In [ ]:
# 병렬 Chain - 병렬 결과 통합

# 병렬적으로 결과 분석
analysis_chain = RunnableParallel(
    pros = ChatPromptTemplate.from_template("{topic}의 장점") | llm | StrOutputParser(),
    cons = ChatPromptTemplate.from_template("{topic}의 단점") | llm | StrOutputParser(),
)

# 결과 통합
synthesis_prompt = ChatPromptTemplate.from_template(
    """다음 분석을 종합하여 결론을 작성하세요:
    장점:
    {pros}

    단점:
    {cons}

    균형 잡힌 결론을 3문장으로 작성하세요."""
)

#전체 chian 만들기. (병렬적으로 결과 분석 -> 통합까지 진행)
full_chain = (
    analysis_chain
    |synthesis_prompt
    |llm
    |StrOutputParser()
)

result = full_chain.invoke({"topic" : "원격 근무"})
print(result)

In [ ]:
# 조건부 분기- RunnableBranch 사용
from langchain_core.runnables import RunnableBranch, RunnableLambda

#언어별 다른 prompt
korean_prompt = ChatPromptTemplate.from_template(
    "다음 한국어 질문에 한국어로 답변하세요: {question}"
)

english_prompt = ChatPromptTemplate.from_template(
    "Answer the following question in English: {question}"
)

default_prompt = ChatPromptTemplate.from_template(
    "Please answer: {question}"
)

#언어 감지 함수
def detect_language(input_dict):
  question = input_dict.get("question", "")
  if any('\uac00' <= char <= '\ud7a3' for char in question):
    #한글이 질문에 있다면
    return "korean"
  return "english"


#조건부 분기
branch_chain = RunnableBranch(
    (lambda x: detect_language(x) == "korean", korean_prompt | llm | StrOutputParser()),
    (lambda x: detect_language(x) == "english", english_prompt | llm | StrOutputParser()),
    default_prompt|llm|StrOutputParser()
    # 기본 동작
)


result_kr = branch_chain.invoke({"question" : "파이썬은 뭐야?"})
result_eng = branch_chain.invoke({"question" : "What is python?"})

print(result_kr)
print(result_eng)

In [ ]:
# RAG 스타일의 multi chain

# 가상의 검색 함수
def retrieve_context(query: str) -> str:
    return f"검색된 context : {query}에 대한 정보..."


#RAG chain 구성
rag_chain = (
    #1. query와 context 병렬 준비
    RunnableParallel(
        question = RunnablePassthrough(),
        context = RunnableLambda(lambda x: retrieve_context(x["question"]))
    )
    #2. prompt 생성
    |ChatPromptTemplate.from_template(
        """context 참고하여 질문에 답변하세요.

        context : {context}
        질문 : {question}
        답변:"""
    )
    |llm
    | StrOutputParser()
)

result = rag_chain.invoke({"question" : "Langchain이란?"})
# rag_chain에서 context는 x = {"question" : "Langchain이란?"}
# -> x["question"] -> "Langchain이란?" -> retrieve_context("Langchain이란?")
# -> retrieve_context에서 "검색된 context: Langchain이란?에 대한 정보... 반환"

# question만 넘기지 않고 context까지 넘기는 이유
# question만 넘기면 LLM은 자기 내부 학습 data만으로 답변을 생성.
# context까지 넘기면 최신 정보로 update할 수 있고 주어진 정보 안에서만 답하도록 만듦.
# context에 기업 문서, 최신 정보 등을 포함하여 외부 지식 기반 답변을 하도록 만듦.
print(result)

LangChain에서 Message


---


Message의 구성 요소
- role(역할) : 메시지의 발신자 유형
- content(내용) : 실제 메시지 내용
- metadata : 부가 정보


---


Langchain의 주요 message 유형
- SystemMessage
  - 모델의 행동 방식을 정의함.
  - 대화의 맥락, 역할, 제약사항 등을 설정.
  - 사용자 입력보다 우선시됨.
- HumanMessage
  - 사용자의 입력(실제 질문/요청)을 나타냄.
  - 텍스트, 이미지 등 multimodal content 포함 가능.
  - 실제 입력 데이터이므로 매 요청마다 바뀜.
- AIMessage
  - 모델의 응답을 나타냄.
  - 모델 호출 결과로 반환되며 다양한 metadata 포함함.
- ToolMessage
  - 도구 실행 결과를 모델에 전달할 때 사용함.



In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

#system message
system_msg = SystemMessage(content="""당신은 파이썬 프로그래밍 전문가입니다.
- 초보자도 이해할 수 있게 설명하세요.
- 코드 예제를 포함하세요.
- 한국어로 답변하세요.""")

#human message (text message)
human_msg = HumanMessage(content = "파이썬의 list comprehension 설명해주세요.")

# 모델 호출 - AIMessage 반환
response = llm.invoke([HumanMessage(content="안녕하세요!")])
print(type(response)) # AIMessage 타입임을 확인할 수 있음.
print(response.content) # 안녕하세요! 어떻게 도와드릴까요?

Streaming(스트리밍)


---

Streaming의 필요성
- 일반 호출(invoke) :전체 응답 완료 후 반환
  => 내부 처리, 배치 작업에 사용.
- 스트리밍(stream) : token 단위 실시간 반환
  => 사용자 인터페이스, 챗봇에 사용.

In [ ]:
# Chain streaming
prompt = ChatPromptTemplate.from_template("{topic}에 대해 설명해주세요.")

chain = prompt | llm | StrOutputParser()

# 토큰 단위 순차 출력됨.
for chunk in chain.stream({"topic" : "머신러닝"}):
  print(chunk, end="", flush=True)

Prompt 작성 원칙
- 명확성 (구체적으로)
- 구체성 (세부 사항과 제약 조건 명시)
- 컨텍스트 (필요한 배경 정보 제공)
- 구조화 (역할, 지시, 예시 분리)
- 출력 형식 (원하는 응답 형식 지정)
- 예시 제공(패턴 학습)
- 환각 방지 (모르는 건 인정하도록 유도)

In [ ]:
# 원칙 1. 명확성

specific_prompt = ChatPromptTemplate.from_template(
    """{topic}에 대해 다음 형식으로 설명해주세요:

    1.정의(1~2문장)
    2. 핵심특징 3가지
    3. 실제 활용 사례 2가지

    전문 용어는 피하고 초보자도 이해할 수 있게 작성해주세요. """
)


# 원칙 2. 구체성

prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 기술 문서 작성 전문가입니다.
    응답 규칙:
    - 문장은 간결하게(20단어 이내)
    - 능동태 사용
    - 전문 용어 사용 시 괄호 안에 설명 추가
    - 총 200자 이내로 작성"""),
    ("human", "{question}")
])

# 원칙 3. 컨텍스트 제공

with_context = ChatPromptTemplate.from_messages([
    ("system", "당신은 시니어 백엔드 개발자입니다."),
    ("human", """다음 프로젝트 요구사항에 맞는 파이썬 웹 프레임워크를 추천해주세요.

프로젝트 정보:
- 팀 규모: 3명 (주니어 2, 시니어 1)
- 예상 트래픽: 일 10만 요청
- 주요 기능: REST API, 실시간 알림
- 기한: 3개월

각 프레임워크의 장단점과 이 프로젝트에 적합한 이유를 설명해주세요.""")
])

# 원칙 4. 구조
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 코드 리뷰 전문가입니다."),
    ("human", """다음 코드를 리뷰해주세요:

<code>
{code}
</code>

다음 순서로 분석해주세요:
1. **기능 요약**: 코드가 수행하는 작업 설명
2. **장점**: 잘 작성된 부분 (2-3가지)
3. **개선점**: 수정이 필요한 부분 (2-3가지)
4. **리팩토링 제안**: 개선된 코드 예시

각 섹션은 명확히 구분하여 작성해주세요.""")
])


# 원칙 5. 출력 형식 지정

# 표 형식 요청
table_prompt = ChatPromptTemplate.from_messages([
    ("system", "응답은 마크다운 표 형식으로 작성하세요."),
    ("human", """다음 프로그래밍 언어들을 비교해주세요: {languages}

| 언어 | 주요 용도 | 장점 | 단점 | 학습 난이도 |
형식으로 작성해주세요.""")
])

# JSON 형식 요청
json_prompt = ChatPromptTemplate.from_messages([
    ("system", "응답은 유효한 JSON 형식으로만 작성하세요. 다른 텍스트는 포함하지 마세요."),
    ("human", """다음 텍스트에서 정보를 추출하세요: {text}

형식:
{{
    "name": "이름",
    "email": "이메일",
    "phone": "전화번호"
}}""")
])

# 번호 목록 형식 요청
list_prompt = ChatPromptTemplate.from_messages([
    ("system", "응답은 번호 목록으로 작성하세요. 각 항목은 한 줄로 간결하게."),
    ("human", "{topic}의 장점 5가지를 알려주세요.")
])



In [ ]:
list_prompt = ChatPromptTemplate.from_messages([
    ("system", "응답은 번호 목록으로 작성하세요. 각 항목은 한 줄로 간결하게."),
    ("human", "{topic}의 장점 5가지를 알려주세요.")
])

chain = list_prompt | llm | StrOutputParser()

result = chain.invoke({"topic" : "linux"})
print(result)

In [ ]:
table_prompt = ChatPromptTemplate.from_messages([
    ("system", "응답은 마크다운 표 형식으로 작성하세요."),
    ("human", """다음 프로그래밍 언어들을 비교해주세요: {languages}

| 언어 | 주요 용도 | 장점 | 단점 | 학습 난이도 |
형식으로 작성해주세요.
각 언어의 답변에 대해서 표의 간격을 잘 맞춰서 어긋나지 않게 해주세요.""")
])

chain = table_prompt | llm | StrOutputParser()

result = chain.invoke({"languages" : {"C", "Java", "Python"}})
print(result)

In [ ]:
# 원칙 7. 환각 방지 (RAG 스타일의 context 기반 응답)

prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 문서 기반 Q&A 어시스턴트입니다.

규칙:
1. 오직 제공된 컨텍스트 내의 정보만 사용하세요
2. 컨텍스트에 없는 정보는 "문서에서 해당 정보를 찾을 수 없습니다"라고 답하세요
3. 답변 시 관련 부분을 인용하세요"""),
    ("human", """컨텍스트:
{context}

질문: {question}

위 컨텍스트만을 참고하여 답변해주세요.""")
])


PromptTemplate(prompt template)

= 단일 문장 / 간단항 명령을 입력하여 단일 문장 / 간단한 응답을 생성하는 데 사용되는 prompt를 구성할 수 있는 문자열 template


---
구성 요소
- 지시 : 언어 모델에게 어떤 작업을 수행하도록 요청하는 구체적인 지시.
- 예시 : 요청된 작업을 수행하는 방법에 대한 한 개 이상의 예시.
- 맥락 : 특정 작업을 수행하기 위한 추가적인 맥락
- 질문 : 어떤 답변을 요구하는 구체적인 질문.


In [ ]:
# prompt 생성의 간단한 과정

#prompt template 정의.
template_text = "안녕하세요, 제 이름은 {name}이고, 나이는 {age}살 입니다."

# PromptTemplate의 instance인 prompt_template 생성
prompt_template = PromptTemplate.from_template(template_text)

#template의 instance에 값을 채워서 prompt 완성.
filled_prompt = prompt_template.format(name = "Hana", age = "24")

filled_prompt


In [ ]:
# PromptTemplate 간의 결합
# 문자열과 PromptTemplate 간의 결합이 모두 가능. (각자끼리도 가능)

combined_prompt = (
    prompt_template
    + PromptTemplate.from_template("\n\n  안녕하세요.")
    + "\n\n 반갑습니다."
)

filled_prompt = combined_prompt.format(name = "Hana", age = "24")
filled_prompt

ChatPromptTemplate
- 대화형 상황에서 여러 메시지 입력을 기반으로 단일 메시지 응답을 생성하는 데 사용됨.
- 대화형 모델 / 챗봇 개발에 주로 사용됨.

In [ ]:
# ChatPromptTemplate.from_messages() 사용.
# 해당 method는 2-tuple 형태의 message list 입력 받아, 각 message의 역할(type)과 내용(content) 기반으로 prompt 구성.

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "이 시스템은 천문학 질문에 답변할 수 있습니다."),
    ("user", "{user_input}"),
])

messages = chat_prompt.format_messages(user_input = "태양계에서 가장 큰 행성은 무엇인가요?")
messages

In [ ]:
chain = chat_prompt | llm | StrOutputParser()

result = chain.invoke({"user_input" : "태양계에서 가장 큰 행성은 무엇인가요?"})
print(result)


RAG - Document Loader

- WebBaseLoader
  - 특정 Web page의 내용을 load하고 parsing하기 위해 설계된 클래스
  - web_paths : web page의 URL을 단일 문자열 / 여러 개의 URL의 시퀀스 배열로 지정.
  - bs_kwargs: BeautifulSoup 사용하여 HTML parsing할 때 사용되는 인자들을 dictionary 형태로 제공함.
- TextLoader
  - text file의 내용을 langchain의 Document 객체로 변환하고 이를 list 형태로 반환함.
- DirectoryLoader
  - directory 내의 모든 문서를 load할 수 있음.
  - instance 생성할 때 문서가 있는 directory의 경로와 해당 문서를 식별할 수 있는 glob 패턴을 지정함.
  - 문서를 읽고 처리하기 위해 내부적으로 UnstructuredLoader가 사용됨. 정의된 형식이 없는 text 문서를 전처리하고 구조화된 형식으로 변환함. (unstructured 라이브러리 설치되어 있어야 함.)
- CSVLoader
  - CSV 파일의 각 행을 추출하여 서로 다른 Document 객체로 변환.
  - 한국어 Windows 환경에서는 주로 encoding 방식으로 'cp949' 사용함. 하지만 오류가 발생해서 'utf-8' 사용하여 load 진행함.
- PDF__Loader
  - PyPDFLoader (pypdf 라이브러리 먼저 설치해야 함.)
  - UnstructuredPDFLoader
  - PyMuPDFLoader
  - OnlinePDFLoader (module-not-found 오류 발생)
  - PyPDFDirectoryLoader


---
Document 객체
- page_content : 텍스트로 변환된 문자열
- metatda : 원본 파일에 대한 metadata


In [ ]:
!pip show langchain_community

In [ ]:
# WebBaseLoader 이용하여 Web Page data 가져오기.

import bs4
from langchain_community.document_loaders import WebBaseLoader

url1 = "https://blog.langchain.dev/customers-replit/"
url2 = "https://blog.langchain.dev/langgraph-v0-2/"

loader = WebBaseLoader(
    web_paths=(url1, url2),
    bs_kwargs = dict(
        parse_only=bs4.SoupStrainer(
            class_=("article-header", "article-content")
            # 해당 class 이름을 가진 HTML 요소만 parsing함.
        )
    ),
)

docs = loader.load()
# docs에는 load된 문서들의 배열이 할당됨. 각 page별로 별도의 document 객체로 변환됨.
# 즉, 2개의 문서가 생성됨.
len(docs)
docs[0]

In [ ]:
# TextLoader 이용하여 Text file data 가져오기.
from langchain_community.document_loaders import TextLoader

loader = TextLoader('dummy_data.txt')
data = loader.load()

print(type(data))
print(len(data))
print(len(data[0].page_content))
print("\n" + data[0].page_content + "\n")
print(data[0].metadata)

In [ ]:
# DirectoryLoader 이용하여 특정 folder의 모든 파일 가져오기.
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(path='/content/dummy_datas', glob="*.txt", loader_cls = TextLoader)
# 지정된 경로와 glob 패턴에 맞는 파일을 찾아 load함.
data = loader.load()

len(data) # load된 file의 개수를 의미함.

print(data[0].page_content + "\n\n")
print(data[1].page_content)

In [ ]:
#PyPDFLoader 이용하여 PDF 파일 데이터 가져오기

from langchain_community.document_loaders import PyPDFLoader

pdf_filepath = '/content/dummy_datas/ch1-intro-db.pdf'
loader = PyPDFLoader(pdf_filepath)
pages = loader.load()

len(pages) # 각 page별로 별개의 Document 객체로 반환되므로 page 수만큼의 길이를 가짐.

print(pages[0].page_content)

In [ ]:
!pip install unstructured unstructured-inference

In [ ]:
!pip install langchain-community pypdf

In [ ]:
#OnlinePDFLoader 이용하여 온라인 PDF 파일의 데이터 가져오기

from langchain_community.document_loaders import OnlinePDFLoader

loader = OnlinePDFLoader("https://arxiv.org/pdf/1706.03762.pdf")
pages = loader.load()

len(pages)

In [ ]:
#PyPDFDirectoryLoader 이용하여 특정 폴더의 모든 PDF 파일 가져오기
from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader('/content/dummy_datas')
data = loader.load()

len(data) # load된 pdf 문서 객체의 총 개수(page별로 반환하기 때문에 모든 file의 page 수의 총합)
print(data[-1].page_content) # 마지막 문서 객체(page) 출력

## TextSplitter

TextSplitter의 주요 동작 : 긴 문서를 작은 단위인 chunk로 나누는 동작. (chunking이라고 부르기도 함.)


---


Chunking이 필요한 이유 : LLM 모델의 입력 token 개수가 정해져 있기 때문. 허용 한도를 넘는 text는 모델에서 입력으로 처리할 수 없게 됨. 핵심 정보가 유지될 수 있는 적절한 크기로 나누는 것이 중요함.


---


TextSplitter 선택하는 기준
- 텍스트가 어떻게 분리되는지(각 chunk가 독립적으로 의미를 갖도록 나눠야 함. 문장, 구절, 단락 등 문서 구조를 기준으로 나눌 수 있음.)
- chuk 크기가 어떻게 측정되는지(LLM 모델의 입력 크기와 비용 등을 종합적으로 고려하여 application에 적합한 최적 크기를 결정하는 기준. 단어 수, 문자 수 등을 기준으로 나눌 수 있음.)


---


LangChain이 제공하는 TextSplitter 도구
- CharacterTextSplitter
  - 주어진 text를 문자 단위로 분할하는 데 사용됨. (python의 split 함수와 유사한 동작)
  - separator : 분할된 각 chunk를 구분할 때 기준이 되는 문자열. ' ' (빈 문자열)로 지정하면 각 글자를 기준으로 분할됨.
  - chunk_size : 각 chunk의 최대 길이.
  - chunk_overlap : 인접한 chunk 사이에 중복으로 포함될 문자의 수. (연결 부분에서 chunk_overlap 수만큼 중복됨.)
  - length_function : chunk의 길이를 계산하는 함수.
  => 각 chunk가 chunk_size 초과하지 않으면서 chunk_overlap만큼 중복되게 함으로써 의미적 연속성을 유지하면서 데이터를 작은 단위로 분할.
- RecursiveCharacterTextSplitter
  - text를 재귀적으로 분할하여 의미적으로 관련 있는 text 조각들이 같이 있도록 하는 목적으로 설계된 class.
  - 문자 list(['\n\n', '\n', ' ', ''])의 문자를 순서대로 사용하여 text를 분할함.
  - 분할한 chunk들이 설정된 chunk_size보다 작아질 때까지 분할 과정을 반복함.



---

Tokenizer 활용.
- LLM 사용할 때 model이 처리할 수 있는 token 수에는 한계가 있음.
- 입력 데이터를 model의 제한을 초과하지 않도록 적절히 분할하는 것이 중요함.
- 이때, LLM 모델에 적용되는 tokenizer를 기준으로 text를 token으로 분할하고, 이 token의 수를 기준으로 text를 chunk로 나누면 model 입력 token 수에 맞춰 분할 가능함.
- OpenAI API는 tiktoken 라이브러리를 통해 해당 모델에서 사용하는 tokenizer 기준으로 분할이 가능함.


In [ ]:
# CharacterTextSplitter 사용 예제

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter


loader = TextLoader('/content/dummy_datas/dummy_data.txt')
data = loader.load()

print(len(data[0].page_content)) # 총 82글자로 이루어진 text file

text_splitter = CharacterTextSplitter(
    separator = '', # 각 글자를 기준으로 분리
    chunk_size = 20,
    chunk_overlap = 4,
    length_function = len, # 문자열의 길이를 기반으로 chunk의 길이를 계산함.
)

texts = text_splitter.split_text(data[0].page_content)

len(texts) # 총 7개의 chunk로 분할되었음.

for i in range(len(texts)):
  print( texts[i] + "\n---------------------------------------\n")

In [ ]:
text_splitter = CharacterTextSplitter(
    separator = '\n', # 줄 바꿈 문자를 기준으로 나누기.
    chunk_size = 32,
    chunk_overlap = 4,
    length_function = len,
)
# 만약, 줄 바꿈 문자를 기준으로 분할할 때 chunk_size보다 커지면 실행은 되지만 Warning 발생!


texts = text_splitter.split_text(data[0].page_content)

print(len(texts))
# 총 3개로 분리됨.
# 줄 바꿈 문자를 기준으로 최대한 chunk_size에 가까운 위치에서 분할.
# 줄 바꿈 문자마다 분할되지 않은 이유가 이것 때문. (chunk_size에 가깝게 분할되기 때문.)
# 주의할 점 : 줄 바꿈 문자가 있는 최대 문자열의 길이보다는 chunk_size가 커야 함.
# 예를 들어, 도핑금지약물정보가 있는 문장은 길이가 김. 줄 바꿈 문자를 기준으로 분할되기 때문에 chunk_size는 이 문자열의 길이보다는 길어야 함.
for i in range(len(texts)):
  print( texts[i] + "\n---------------------------------------\n")


In [ ]:
# RecursiveCharacterTextSplitter 사용하는 예제

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 20,
    chunk_overlap = 4,
    length_function = len,
)

texts = text_splitter.split_text(data[0].page_content)

print(len(texts))

for i in range(len(texts)):
  print( texts[i] + "\n---------------------------------------\n")

# chunk_overlap 변경해도 chunk에 변화는 없음.
# 이어지는 한 개의 문자열에 대해서는 끊김 없이 한 개의 chunk로 분할됨.
# chunk_size에 가깝거나 그보다 작게 분할됨.

In [ ]:
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 40,
    chunk_overlap = 10,
    encoding_name = 'cl100k_base'
    # text를 token으로 변환하는 encoding 방식을 의미함.
)

docs = text_splitter.split_documents(data)
docs_content = list()
print(len(docs))

for i in range(len(docs)):
  print(docs[i].page_content + "\n-------------------------\n")
  docs_content.append(docs[i].page_content)


Embedding model(bge-m3) & Vector DB (Chroma)

In [ ]:
!pip install -U sentence-transformers

In [ ]:
import numpy as np
from numpy import dot
from numpy.linalg import norm

def cos_sim(A,B):
  return dot(A,B) / (norm(A)*norm(B))


In [ ]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

embeddings_model = HuggingFaceBgeEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device' : 'cpu'},
    encode_kwargs={'normalize_embeddings':True},
    )


In [ ]:
embeddings = embeddings_model.embed_documents(docs_content)
# docs 그대로 못 넣음. docs는 Document 형태. 해당 method는 list 필요로 함.
# docs의 page_content만 넣은 docs_content라는 list 만들어서 넣음.

len(embeddings), len(embeddings[0])

#len(embeddings) : 텍스트의 개수
# len(embeddings[0]) : dimension(1024. bge-m3의 dimension이 1024임.)

In [ ]:
print(embeddings[0])

In [ ]:

embedded_query = embeddings_model.embed_query('경기 내에 먹을 수 있는 약인가요?')

for embedding in embeddings:
  print(cos_sim(embedding, embedded_query))

# 경기 관련된 내용이 포함된 chunk는 2번째 chunk. 유사도가 가장 높은 것을 확인할 수 있음.
# 아마 3번째에는 약이라는 단어가 포함되어서 그 다음으로 높은 유사도를 가지는 것으로 판단됨.

In [ ]:
!pip install chromadb

In [ ]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_texts(
    docs_content,
    embeddings_model,
    collection_name = 'medicines',
    persist_directory = './db/chromadb',
    collection_metadata = {'hnsw:space' : 'cosine'},
)
# 위 코드와 마찬가지로 Document 객체 자체를 넣을 수 없음.
# CharacterTextSplitter, RecursiveCharacterTextSplitter 사용한 결과는 list이므로 바로 넣을 수 있음.
# 근데 Tokenizer 사용한 경우에는 결과가 Document 객체이므로 따로 넣어야 함.
# Document 객체인 docs의 page_content만 저장한 docs_content라는 list 넣어서 해결!
# 저장소 이름 : medicines
# 저장되는 directory : ./db/chromadb
# 유사도 계산 방법으로는 cosine 유사도 방법을 사용함.

In [ ]:
query = '경기 내에 먹을 수 있는 약인가요?'
most_doc = db.similarity_search(query)
print(most_doc[0].page_content)

In [ ]:
db = Chroma.from_texts(
    texts, #RecursiveCharacterTextSplitter 사용하여 chunking한 결과
    embeddings_model,
    collection_name = 'medicines',
    persist_directory = './db/chromadb',
    collection_metadata = {'hnsw:space' : 'cosine'},
)


In [ ]:
query = '경기 내에 먹을 수 있는 약인가요?'
most_doc = db.similarity_search(query)
print(most_doc[0].page_content)